# Extract GT right-arm trajectory

From a Task 2 demonstration
([`hermanprawiro/task2_fixpos_200`](https://huggingface.co/datasets/hermanprawiro/task2_fixpos_200)),
reconstruct the **right-arm** world-frame path via FK
(`camelo.control.kinematics`). Open-loop joint replay fails when the live
base is not on the demo `(x, y, yaw)`; a world-frame TCP path can still be
tracked with IK from the actual base.

Two right-arm frames (do not mix them):

- **`right_tcp` (TCP)** — gripper tool-centre, at the URDF
  `right_tcp_joint` origin past the gripper base (read from the Lula
  URDF, not hardcoded). This is the RMPflow / Lula end-effector.
  **Saved GT and later IK use this.** The pad is grasped at the TCP, so
  tracking flange joints would miss by a gripper length.
- **`right_fr3v2_link8` (link8 / flange)** — FR3 wrist flange. This is
  what the recorder writes as `observation.state[7:14]`
  (`/isaac/right_ee_pose`). Used only as a sanity check: FK of measured
  joints should match that recorded pose to millimetres.

Set `EPISODE_IDX` below (default 0). Kernel: numpy / pandas / matplotlib
(lerobot-arena is fine). Measured joints, not commanded actions.
Each code cell has a five-point note above it (what + why).


### Interpreter and repo path

- Import numpy / matplotlib only — no ROS, no Isaac, no lerobot.
- Pin `ROOT` to the camelo-ebim checkout so dataset paths are absolute.
- Put `ROOT` and `scripts/debugging` on `sys.path` (`camelo`, `gt_traj_utils`).
- Print `sys.executable` to confirm the kernel (lerobot-arena is fine).
- Why: this notebook is offline. Everything later reads local parquet.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("/home/ubuntu/workspace/camelo-ebim")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DEBUG = ROOT / "scripts" / "debugging"
if str(DEBUG) not in sys.path:
    sys.path.insert(0, str(DEBUG))

print(f"python={sys.executable}")
print(f"root={ROOT}")


### Imports and knobs

- Import `camelo.contracts` for the 37-dim state slices (base, spine, EE).
- Import kinematics + `gt_traj_utils` (parquet load and the mobile-base plot).
- Dataset defaults match `prepare_replay_actions.py` / the replay adapter.
- `EPISODE_IDX` is the user knob (default 0). Change it here, then re-run.
- Print repo / episode / paths so a stale or missing cache is obvious.


In [ ]:
from gt_traj_utils import ensure_tabular, load_episode, plot_mobile_base_pose

from camelo import contracts as C
from camelo.control.gt_traj import compute_gt_traj, write_gt_traj
from camelo.control.kinematics import MobileFR3Kinematics, quat_xyzw_angle_deg
from camelo.policy.adapters.replay import (
    DEFAULT_ACTIONS_DIR,
    DEFAULT_DATASET_DIR,
    DEFAULT_REPO_ID,
    default_gt_traj_path,
)

# --- knobs ---
EPISODE_IDX = 75
REPO_ID = DEFAULT_REPO_ID
DATASET_DIR = ROOT / DEFAULT_DATASET_DIR
OUT_DIR = ROOT / DEFAULT_ACTIONS_DIR
PLOT_EVERY_N = 20
HEADING_TICK_M = 0.15

print(f"repo={REPO_ID}")
print(f"episode={EPISODE_IDX}")
print(f"dataset={DATASET_DIR}")
print(f"out={OUT_DIR}")


### Load one demonstration episode

- Reuse cached `meta/` + `data/` via `gt_traj_utils.ensure_tabular`.
- If missing, `snapshot_download` those prefixes only — no videos.
- Gated hub: accept the dataset terms and export `HF_TOKEN` if download fails.
- Filter parquet to `EPISODE_IDX` and sort by `frame_index`.
- Why: proprio lives in parquet. `LeRobotDataset` would also pull videos.


In [ ]:
ensure_tabular(REPO_ID, DATASET_DIR)
ep, fps = load_episode(DATASET_DIR, EPISODE_IDX)
print(f"frames={len(ep)}  fps={fps:g}  duration={len(ep) / fps:.1f} s")


### Unpack measured state

- Stack `observation.state` to `(T, 37)` and assert `STATE_DIM`.
- Time `t = frame_index / fps` (sim seconds; recorder is 30 Hz).
- Slice base odom `(x, y, yaw)`, spine, right arm joints via `camelo.contracts`.
- Slice recorded `right_ee` into xyz + xyzw quat (`S_RIGHT_EE` = link8, world).
- Why: GT is where the EE actually was. Commanded `action` joints can differ.


In [ ]:
state = np.stack(
    [np.asarray(row, dtype=np.float64) for row in ep["observation.state"]]
)
if state.shape[1] != C.STATE_DIM:
    raise ValueError(f"expected state dim {C.STATE_DIM}, got {state.shape}")

t = ep["frame_index"].to_numpy(dtype=np.float64) / fps
base_xy_yaw = state[:, C.S_BASE_ODOM].copy()
spine = state[:, C.S_SPINE].copy()
arm_q = state[:, C.S_RIGHT_ARM].copy()
recorded_link8 = state[:, C.S_RIGHT_EE].copy()
rec_xyz = recorded_link8[:, :3]
rec_quat = recorded_link8[:, 3:7]

print(
    f"base0 x={base_xy_yaw[0, 0]:.4f} y={base_xy_yaw[0, 1]:.4f} "
    f"yaw={np.degrees(base_xy_yaw[0, 2]):.2f} deg"
)
print(f"spine0={spine[0]:.4f} m  spine median={np.median(spine):.4f} m")
print(
    f"recorded right_ee z: "
    f"min={rec_xyz[:, 2].min():.3f} median={np.median(rec_xyz[:, 2]):.3f} "
    f"max={rec_xyz[:, 2].max():.3f} m"
)


### Plot mobile base pose

- `gt_traj_utils.plot_mobile_base_pose`: top-down `x–y`; start green, end gold.
- Axes box is square; x and y are scaled independently so a small-x
  path still fills the panel (not `equal` aspect).
- Heading ticks every `PLOT_EVERY_N` samples (yaw of the mobile base).
- Time series of `x`, `y`, and yaw so drift vs the demo start is visible.
- Why: this is the disturbance joint replay cannot absorb — the live base
  will not sit on this curve, so EE must be tracked in world coordinates.


In [ ]:
fig_b, _axes = plot_mobile_base_pose(
    t,
    base_xy_yaw,
    title=f"episode {EPISODE_IDX:03d} — mobile base pose",
    plot_every_n=PLOT_EVERY_N,
    heading_tick_m=HEADING_TICK_M,
)
plt.show()
bx, by, byaw = base_xy_yaw[:, 0], base_xy_yaw[:, 1], base_xy_yaw[:, 2]
print(
    f"base Δxy={np.hypot(bx[-1] - bx[0], by[-1] - by[0]) * 1000:.4f} mm  "
    f"Δyaw={np.degrees(byaw[-1] - byaw[0]):+.3f} deg"
)


### Forward-kinematics over the episode

- Call `compute_gt_traj` (same writer prepare/launch use) for TCP + link8 FK.
- TCP is the world GT later IK will track; link8 is the recorded-flange check.
- Use measured joints / spine / base from state, not commanded `action`.
- Frame-0 logs spell out tcp / link8_fk / rec so the three poses are not
  confused (TCP is the GT; rec is USD flange, not TCP).


In [ ]:
def fmt_xyz_mm(xyz) -> str:
    mm = np.asarray(xyz, dtype=float).reshape(3) * 1000.0
    return f"x={mm[0]:+.4f} y={mm[1]:+.4f} z={mm[2]:+.4f} mm"


kin = MobileFR3Kinematics()
gt = compute_gt_traj(
    t=t,
    base_xy_yaw=base_xy_yaw,
    spine=spine,
    arm_q=arm_q,
    recorded_link8=recorded_link8,
    episode=EPISODE_IDX,
    fps=fps,
    kin=kin,
)
tcp_xyz = gt["tcp_xyz"]
tcp_quat = gt["tcp_quat_xyzw"]
link8_xyz = gt["link8_xyz"]
link8_quat = gt["link8_quat_xyzw"]
tcp_joint_xyz = kin.tcp_joint_xyz()
tcp_joint_m = float(np.linalg.norm(tcp_joint_xyz))
print(f"urdf={kin.urdf_path}")
print("frame 0 world xyz — three different frames:")
print("  tcp      FK of right_tcp (gripper TCP; saved GT / IK target)")
print(f"           {fmt_xyz_mm(tcp_xyz[0])}")
print("  link8_fk FK of right_fr3v2_link8 (flange from joints + base)")
print(f"           {fmt_xyz_mm(link8_xyz[0])}")
print("  rec      recorded state[7:14] (USD link8 flange, not TCP)")
print(f"           {fmt_xyz_mm(rec_xyz[0])}")
print(
    f"URDF right_tcp_joint origin (gripper-base) {fmt_xyz_mm(tcp_joint_xyz)}"
)
print(f"           ||xyz||={tcp_joint_m * 1000:.4f} mm")
print(
    f"link8_fk − rec (frame 0) {fmt_xyz_mm(link8_xyz[0] - rec_xyz[0])}  "
    f"|d|={np.linalg.norm(link8_xyz[0] - rec_xyz[0]) * 1000:.4f} mm"
)
print(
    "x/y residuals are sub-mm; z is typically ~1–2 mm (USD root / wheel "
    "settle). Not the URDF tcp-joint origin above."
)


### Sanity-check FK link8 vs recorded flange

- Position error: FK `right_fr3v2_link8` minus recorded `state[7:14]` xyz.
- Median xyz bias: a near-constant offset is USD root / wheel settle.
- Rotation error in degrees (xyzw geodesic); should be << 1° if frames match.
- TCP–link8 distance must match `kin.tcp_joint_xyz()` (URDF), not collapse.
- Pass bar is median position error ≤ 5 mm. Print PASS or FAIL; do not
  absorb any bias into the saved GT.


In [ ]:
pos_err = np.linalg.norm(link8_xyz - rec_xyz, axis=1)
pos_bias = np.median(link8_xyz - rec_xyz, axis=0)
rot_err = np.array(
    [
        quat_xyzw_angle_deg(a, b)
        for a, b in zip(link8_quat, rec_quat, strict=True)
    ]
)
tcp_link8 = np.linalg.norm(tcp_xyz - link8_xyz, axis=1)
PASS_MM = 5.0
median_mm = float(np.median(pos_err) * 1000)
bias_norm = float(np.linalg.norm(pos_bias))
p95_mm = float(np.percentile(pos_err, 95) * 1000)
max_mm = float(pos_err.max() * 1000)
tcp_link8_mm = float(np.median(tcp_link8) * 1000)

print("FK link8 vs recorded right_ee (do not absorb bias into GT)")
print(
    f"  pos err   median={median_mm:.4f} mm  "
    f"p95={p95_mm:.4f} mm  max={max_mm:.4f} mm"
)
print(f"  bias      {fmt_xyz_mm(pos_bias)}")
print(
    f"  rot err deg median={np.median(rot_err):.4f}  "
    f"p95={np.percentile(rot_err, 95):.4f}  max={rot_err.max():.4f}"
)
print(
    f"  TCP–link8 offset  median={tcp_link8_mm:.4f} mm  "
    f"(URDF {fmt_xyz_mm(tcp_joint_xyz)})"
)
if median_mm <= PASS_MM:
    print(
        f"PASS: median flange error {median_mm:.4f} mm ≤ {PASS_MM:.4f} mm"
    )
else:
    print(
        f"FAIL: median flange error {median_mm:.4f} mm > {PASS_MM:.4f} mm — "
        "check URDF / spine / base z"
    )
if bias_norm > 0.005 and bias_norm > 0.7 * float(np.median(pos_err)):
    print(
        f"note: error is mostly a constant xyz bias "
        f"({fmt_xyz_mm(pos_bias)}); "
        "likely USD root / wheel settle. Saved GT is uncorrected FK."
    )


### Plot EE trajectories

- 3D path: recorded link8, FK link8, and TCP GT on the same axes.
- Time series of x / y / z so an xyz bias is visible at a glance.
- FK-vs-recorded error (mm and deg) vs time — should stay ~mm, not jump.
- TCP–link8 offset vs time; dashed line is `kin.tcp_joint_xyz()` from URDF.
- Why: confirm the GT looks like the demo before writing the npz.


In [ ]:
fig_e = plt.figure(figsize=(16, 8.5))
ax3 = fig_e.add_subplot(2, 2, 1, projection="3d")
ax3.plot(*rec_xyz.T, color="0.55", lw=1.2, label="recorded link8")
ax3.plot(*link8_xyz.T, color="tab:orange", lw=1.0, label="FK link8")
ax3.plot(*tcp_xyz.T, color="tab:blue", lw=1.4, label="FK TCP (GT)")
ax3.scatter(*tcp_xyz[0], c="green", s=30)
ax3.scatter(*tcp_xyz[-1], c="gold", s=30)
ax3.set_xlabel("x (m)")
ax3.set_ylabel("y (m)")
ax3.set_zlabel("z (m)")
ax3.set_title("right EE world path")
ax3.legend(loc="best", fontsize=8)

ax_xyz = fig_e.add_subplot(2, 2, 2)
for arr, label, color in zip(
    (tcp_xyz, link8_xyz, rec_xyz),
    ("TCP GT", "FK link8", "recorded link8"),
    ("tab:blue", "tab:orange", "0.5"),
    strict=True,
):
    ax_xyz.plot(t, arr[:, 0], color=color, lw=1.0, label=f"{label} x")
    ax_xyz.plot(t, arr[:, 1], color=color, lw=1.0, ls="--")
    ax_xyz.plot(t, arr[:, 2], color=color, lw=1.0, ls=":")
ax_xyz.set_xlabel("t (s)")
ax_xyz.set_ylabel("m")
ax_xyz.set_title("EE x / y / z vs time")
ax_xyz.legend(loc="best", fontsize=7, ncol=3)
ax_xyz.grid(True, alpha=0.3)

ax_err = fig_e.add_subplot(2, 2, 3)
ax_err.plot(t, pos_err * 1000, color="tab:red", lw=1.0, label="|FK−rec| xyz")
ax_err.axhline(PASS_MM, color="0.35", ls="--", lw=0.9, label=f"pass ≤ {PASS_MM:.0f} mm")
ax_err.plot(t, rot_err, color="tab:purple", lw=1.0, label="rot err (deg)")
ax_err.set_xlabel("t (s)")
ax_err.set_ylabel("mm  /  deg")
ax_err.set_title("FK link8 vs recorded flange")
ax_err.legend(loc="best", fontsize=8)
ax_err.grid(True, alpha=0.3)

ax_off = fig_e.add_subplot(2, 2, 4)
ax_off.plot(t, tcp_link8, color="tab:blue", lw=1.0)
ax_off.axhline(
    tcp_joint_m,
    color="0.4",
    ls="--",
    lw=0.8,
    label=f"URDF ||tcp joint||={tcp_joint_m * 1000:.4f} mm",
)
ax_off.set_xlabel("t (s)")
ax_off.set_ylabel("m")
ax_off.set_title("TCP–link8 offset (must not collapse to 0)")
ax_off.legend(loc="best", fontsize=8)
ax_off.grid(True, alpha=0.3)

fig_e.suptitle(f"episode {EPISODE_IDX:03d} — right-arm GT vs recorded flange")
fig_e.tight_layout()
plt.show()


### Save GT trajectory

- Write `outputs/replay/task2_fixpos_200/epXXX_gt_traj.npz` next to replay actions.
- Store `t`, base pose, spine, right `arm_q`, TCP, FK link8, and recorded link8.
- Quaternions are xyzw, matching `observation.state` / the dataset contract.
- TCP (`tcp_xyz`, `tcp_quat_xyzw`) is the GT step-2 IK will track.
- Recorded link8 stays in the file so later audits do not need the parquet.


In [ ]:
out_path = ROOT / default_gt_traj_path(EPISODE_IDX)
write_gt_traj(out_path, gt)
print(f"wrote {out_path}")
print(
    f"  T={len(t)}  keys={sorted(np.load(out_path).files)}  "
    f"TCP is the GT for step-2 IK"
)
